# CV Scan

In [1]:
import pymupdf
import base64
import requests
import json
import os
from pathlib import Path
import time
import boto3
from botocore.exceptions import ClientError
from dotenv import load_dotenv
load_dotenv()

True

In [5]:
def process_and_send(pdf_path: str, url: str):
    path = Path(pdf_path)
    if not path.exists():
        print(f"Error: File not found at {path}")
        return

    print(f"Opening {path.name}...")
    doc = pymupdf.open(path)
    images_base64 = []

    # 1. Convert each page to a base64 PNG
    for page_num in range(len(doc)):
        page = doc.load_page(page_num)
        # dpi=150 is a good balance between image quality for Textract and payload size
        pix = page.get_pixmap(dpi=150) 
        image_bytes = pix.tobytes("png")
        
        # Base64 encode and decode to utf-8 string for JSON serialization
        b64_string = base64.b64encode(image_bytes).decode('utf-8')
        images_base64.append(b64_string)

    print(f"Converted {len(images_base64)} pages. Sending to Lambda...")

    # 2. Prepare payload and send POST request
    payload = {
        "images_base64": images_base64
    }
    
    headers = {
        "Content-Type": "application/json"
    }

    try:
        response = requests.post(url, json=payload, headers=headers)
        response.raise_for_status()  # Check for HTTP errors
        
        # 3. Print the results
        result_data = response.json()
        print("\n--- Lambda Response ---")
        print(json.dumps(result_data, indent=2))
        
    except requests.exceptions.RequestException as e:
        print(f"\nRequest failed: {e}")
        if response is not None:
            print(f"Response Body: {response.text}")

In [ ]:
PDF_PATH = "Nitay_Levy_MLN_CV.pdf"
CV_SCAN_LAMBDA_URL = os.environ.get("CV_SCAN_LAMBDA_URL", "https://your-api-id.execute-api.region.amazonaws.com/prod/analyze")
process_and_send(PDF_PATH, CV_SCAN_LAMBDA_URL)

# Scan Job Posts and Store on S3

In [ ]:
REGION = os.environ.get("AWS_DEFAULT_REGION", "eu-west-1")
SQS_QUEUE_URL = os.environ.get("JOB_SCAN_SQS_URL", "https://sqs.eu-west-1.amazonaws.com/123456789012/job-parsing-queue")
BUCKET_NAME = "scanned-jobs"

sqs_client = boto3.client("sqs", region_name=REGION)
s3_client = boto3.client("s3", region_name=REGION)

# 1. Payload with two distinct jobs
jobs_to_scan = [
    {
        "job_id": "JOB-101",
        "job_company": "AcmeCorp",
        "job_description": """
About AcmeCorp:
We are hiring a Senior Data Engineer.
Responsibilities:
- Build resilient ETL pipelines using PySpark and SQL.
Requirements:
- 5+ years of Python software development.
- B.Sc. in Computer Science or equivalent.
Nice to have:
- Hands-on experience with AWS Bedrock or LLMs.
        """
    },
    {
        "job_id": "JOB-102",
        "job_company": "BetaTech",
        "job_description": """
BetaTech is looking for a Backend Engineer.
Role Description:
- Develop microservices and maintain APIs.
Qualifications:
- 3+ years experience with Node.js / Go.
- Strong relational database experience (PostgreSQL).
Bonus:
- Experience with Kubernetes and Docker.
        """
    }
]

# 2. Send both jobs to SQS in a single batch
def send_jobs_to_sqs(jobs: list):
    entries = [
        {
            "Id": f"job_entry_{idx}",
            "MessageBody": json.dumps(job)
        }
        for idx, job in enumerate(jobs)
    ]
    
    response = sqs_client.send_message_batch(
        QueueUrl=SQS_QUEUE_URL,
        Entries=entries
    )
    
    print(f"Sent {len(response.get('Successful', []))} messages to SQS.")
    if response.get("Failed"):
        print(f"Failed messages: {response.get('Failed')}")

# 3. Poll S3 until the files are written
def fetch_scanned_job_from_s3(company: str, job_id: str, timeout_sec: int = 120, poll_interval: int = 5):
    """
    Polls S3 for the file '{company}/{job_id}.json' as created by the Lambda.
    """
    s3_key = f"{company}/{job_id}.json"
    start_time = time.time()
    
    print(f"Waiting for s3://{BUCKET_NAME}/{s3_key}...")
    
    while time.time() - start_time < timeout_sec:
        try:
            response = s3_client.get_object(Bucket=BUCKET_NAME, Key=s3_key)
            content = response["Body"].read().decode("utf-8")
            return json.loads(content)
        except ClientError as e:
            # S3 returns NoSuchKey (404) until Lambda finishes and uploads the object
            if e.response["Error"]["Code"] == "NoSuchKey":
                time.sleep(poll_interval)
            else:
                raise e
                
    raise TimeoutError(f"Timed out waiting for {s3_key} after {timeout_sec}s.")

In [ ]:
# Step A: Enqueue jobs
send_jobs_to_sqs(jobs_to_scan)

# Step B: Retrieve both results once processed
scanned_results = []
for job in jobs_to_scan:
    company = job["job_company"].strip()
    job_id = job["job_id"].strip()
    
    try:
        record = fetch_scanned_job_from_s3(company, job_id, timeout_sec=120)
        scanned_results.append(record)
        print(f"Successfully retrieved {job_id} from S3!")
    except TimeoutError as err:
        print(err)

# Output consolidated parsed JSON
print("\n--- Final Consolidated Scanned Jobs ---")
print(json.dumps(scanned_results, indent=2, ensure_ascii=False))

Waiting for s3://daily-scanned-jobs/AcmeCorp/JOB-101.json...
Successfully retrieved JOB-101 from S3!
Waiting for s3://daily-scanned-jobs/BetaTech/JOB-102.json...
Successfully retrieved JOB-102 from S3!

--- Final Consolidated Scanned Jobs ---
[
  {
    "job_id": "JOB-101",
    "job_company": "AcmeCorp",
    "responsibilities": "Responsibilities:\n- Build resilient ETL pipelines using PySpark and SQL.",
    "Degree Types": "[B.Sc.] [Quote: \"B.Sc. in Computer Science or equivalent.\"]",
    "Degree Names": "[Computer Science] [Quote: \"Computer Science\"]",
    "Accepting Related Degrees or Experience": "Yes [Quote: \"equivalent\"]",
    "University Only": "No [Quote: \"None\"]",
    "Remaining Studies": "None [Quote: \"None\"]",
    "Grade": "None [Quote: \"None\"]",
    "Availability": "None",
    "Min Experience Years 1": "5+ [Quote: \"5+ years of Python software development.\"]",
    "Soft Skills": "None",
    "Advantage - Degree Types": "None [Quote: \"None\"]",
    "Advantage - De

In [5]:
import boto3
import json
import random

def display_random_jobs_full_info():
    # Initialize the S3 client
    s3 = boto3.client('s3')
    
    bucket_name = 'weekly-jobs'
    file_key = 'exports/latest_jobs.json'
    
    try:
        # Fetch the JSONL file from S3
        response = s3.get_object(Bucket=bucket_name, Key=file_key)
        
        # Read and decode the file content
        file_content = response['Body'].read().decode('utf-8')
        
        # Parse the JSON Lines (.jsonl) into a list of dictionaries
        jobs = []
        for line in file_content.splitlines():
            line = line.strip()
            if line:
                jobs.append(json.loads(line))
        
        if not jobs:
            print("No jobs found in the file.")
            return

        # Pick 5 random jobs (or fewer if the file has less than 5 jobs)
        sample_size = min(5, len(jobs))
        random_jobs = random.sample(jobs, sample_size)
        
        # Display ALL information nicely
        for i, job in enumerate(random_jobs, start=1):
            print(f"{'='*25} JOB {i} {'='*25}")
            
            # json.dumps with indent=4 formats all nested arrays and keys beautifully
            formatted_job = json.dumps(job, indent=4, ensure_ascii=False)
            print(formatted_job)
            
            print("\n")
            
    except Exception as e:
        print(f"Error retrieving or parsing jobs from S3: {e}")

# Run the function
if __name__ == "__main__":
    display_random_jobs_full_info()

========================= JOB 1 =========================
[
    {
        "job_id": "58.479",
        "company": "Claroty",
        "title": "Backend Platform Engineer",
        "location": "Tel Aviv, IL",
        "url": "https://www.comeet.com/jobs/Claroty/F2.004/backend-platform-engineer/58.479",
        "employment_type": "Full Time Employee",
        "updated_at_formatted": "2026-11-14 07:18 UTC",
        "content_full": "Description\nWe’re growing and looking to hire a Platform Engineer who embodies our core values: People First, Customer Obsession, Strive for Excellence, and Integrity.\nAbout Claroty:\nClaroty is on a mission to secure cyber-physical systems across industrial, healthcare, commercial and public sector environments: the Extended Internet of Things (XIoT). The Claroty Platform integrates with customers’ existing infrastructure to provide a full range of controls for visibility, exposure management, network protection, threat detection, and secure access. Our solutio